# Chapter 70 — Cross-Entropy Loss for GPT Outputs

Chapter 69 created one shifted target token for every position in a GPT training batch.

This chapter turns class-last GPT logits and those targets into one scalar cross-entropy loss.


## Learning goals

By the end of this chapter, you will be able to:

- explain the `[B, T, V]` GPT logit shape;
- align each vocabulary-logit vector with one target token ID;
- interpret one-position cross-entropy as negative log-probability;
- reshape logits to `[B × T, V]` and targets to `[B × T]`;
- compute mean loss over all positions;
- compare flattening with PyTorch's higher-rank class-second form;
- implement a validated reusable loss helper; and
- diagnose common dtype, range, class-dimension, and softmax mistakes.


## Terms and shape contract

A **logit** is a raw, unnormalized score for one class.

A **class** is one possible output label, which is one vocabulary token in next-token prediction.

A **target token ID** is the integer index of the correct vocabulary class.

Cross-entropy measures the negative log-probability assigned to the correct class.

For GPT training:

| Tensor | Shape |
|---|---|
| Vocabulary logits | `[B, T, V]` |
| Target token IDs | `[B, T]` |
| Flattened logits | `[B × T, V]` |
| Flattened targets | `[B × T]` |
| Mean loss | scalar |

Flattening combines only batch and context dimensions while preserving vocabulary as the class dimension.


## Create deterministic GPT-shaped tensors

We will use fake logits to isolate loss behavior from model architecture.

The two batch rows and four positions create eight classification examples over ten vocabulary classes.


In [1]:
import torch

device = "cpu"
logit_generator = torch.Generator(device=device).manual_seed(70)
batch_size = 2
context_length = 4
vocabulary_size = 10
vocabulary_logits = torch.randn(
    batch_size,
    context_length,
    vocabulary_size,
    generator=logit_generator,
    device=device,
)
target_token_ids = torch.tensor(
    [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
    ],
    dtype=torch.long,
    device=device,
)

print("device:", device)
print("logits shape:", vocabulary_logits.shape)
print("targets shape:", target_token_ids.shape)
print("prediction count:", batch_size * context_length)
print("target token IDs:")
print(target_token_ids)

device: cpu
logits shape: torch.Size([2, 4, 10])
targets shape: torch.Size([2, 4])
prediction count: 8
target token IDs:
tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])


`vocabulary_logits[batch_index, position]` is the score vector whose correct class is `target_token_ids[batch_index, position]`.


## Cross-entropy at one position

Softmax converts one position's logits into probabilities.

Taking logarithms produces log-probabilities, and negating the target class's log-probability produces cross-entropy for that position.

PyTorch combines log-softmax and target selection in a numerically stable operation.


In [2]:
selected_batch_index = 0
selected_position = 2
one_position_logits = vocabulary_logits[selected_batch_index, selected_position]
one_target_id = target_token_ids[selected_batch_index, selected_position]
log_probabilities = torch.log_softmax(one_position_logits, dim=-1)
manual_position_loss = -log_probabilities[one_target_id]
pytorch_position_loss = torch.nn.functional.cross_entropy(
    one_position_logits.unsqueeze(0),
    one_target_id.unsqueeze(0),
)
target_probability = log_probabilities[one_target_id].exp()

print("one-position logits shape:", one_position_logits.shape)
print("target token ID:", one_target_id.item())
print("target probability:", target_probability.item())
print("manual negative log-probability:", manual_position_loss.item())
print("PyTorch cross-entropy:", pytorch_position_loss.item())
print("manual and PyTorch results match:", end=" ")
print(torch.allclose(manual_position_loss, pytorch_position_loss))

one-position logits shape: torch.Size([10])
target token ID: 3
target probability: 0.007167713716626167
manual negative log-probability: 4.938168525695801
PyTorch cross-entropy: 4.938168525695801
manual and PyTorch results match: True


A larger probability for the correct target produces a smaller negative log-probability.

The model should pass raw logits to `cross_entropy`, not probabilities, because PyTorch performs the stable log-softmax internally.


## Flatten batch and context dimensions

GPT stores the class dimension last, but the standard two-dimensional cross-entropy form expects `[number_of_examples, number_of_classes]`.

Combining `B` and `T` produces `B × T` examples without changing vocabulary width `V`.

`reshape` is used because it handles compatible non-contiguous layouts more flexibly than `view`.


In [3]:
flattened_logits = vocabulary_logits.reshape(
    batch_size * context_length, vocabulary_size
)
flattened_targets = target_token_ids.reshape(batch_size * context_length)

print("before flattening")
print("  logits:", vocabulary_logits.shape)
print("  targets:", target_token_ids.shape)
print("after flattening")
print("  logits:", flattened_logits.shape)
print("  targets:", flattened_targets.shape)

before flattening
  logits: torch.Size([2, 4, 10])
  targets: torch.Size([2, 4])
after flattening
  logits: torch.Size([8, 10])
  targets: torch.Size([8])


The eight flattened rows still appear in batch-major, position-minor order.

No example or class score is removed.


## Inspect the flattened mapping

For row-major flattening, `flat_index = batch_index × T + position`.

The reverse mapping uses integer division and remainder.


In [4]:
print("flat | batch | position | target ID | logits match")
print("-" * 58)
for flat_index in range(flattened_targets.numel()):
    batch_index = flat_index // context_length
    position = flat_index % context_length
    logits_match = torch.equal(
        flattened_logits[flat_index],
        vocabulary_logits[batch_index, position],
    )
    print(
        f"{flat_index:>4} | "
        f"{batch_index:>5} | "
        f"{position:>8} | "
        f"{flattened_targets[flat_index].item():>9} | "
        f"{str(logits_match):>12}"
    )

flat | batch | position | target ID | logits match
----------------------------------------------------------
   0 |     0 |        0 |         1 |         True
   1 |     0 |        1 |         2 |         True
   2 |     0 |        2 |         3 |         True
   3 |     0 |        3 |         4 |         True
   4 |     1 |        0 |         5 |         True
   5 |     1 |        1 |         6 |         True
   6 |     1 |        2 |         7 |         True
   7 |     1 |        3 |         8 |         True


Every flattened logit row matches its original batch-position vector exactly.


## Compute all-position loss

The default cross-entropy reduction is `mean`.

Using `reduction="none"` first exposes one loss for every flattened example.


In [5]:
individual_losses = torch.nn.functional.cross_entropy(
    flattened_logits,
    flattened_targets,
    reduction="none",
)
mean_loss = torch.nn.functional.cross_entropy(flattened_logits, flattened_targets)
loss_per_position = individual_losses.reshape(batch_size, context_length)

print("individual loss shape:", individual_losses.shape)
print("loss per position shape:", loss_per_position.shape)
print("loss per position:")
print(loss_per_position)
print("mean of individual losses:", individual_losses.mean().item())
print("default mean loss:", mean_loss.item())
print("means match:", torch.allclose(individual_losses.mean(), mean_loss))

individual loss shape: torch.Size([8])
loss per position shape: torch.Size([2, 4])
loss per position:
tensor([[3.7347, 2.2262, 4.9382, 1.5444],
        [2.5760, 1.7429, 3.8100, 4.7006]])
mean of individual losses: 3.159121513366699
default mean loss: 3.159121513366699
means match: True


The scalar loss averages all eight batch-position contributions.

Reshaping unreduced losses back to `[B, T]` is useful for inspection, while training normally backpropagates the scalar mean.


## A valid higher-rank alternative

PyTorch cross-entropy is not limited to two-dimensional scores.

It also accepts scores shaped `[N, C, d1, ...]`, where the class dimension `C` must be second.

GPT logits are `[B, T, V]`, so transposing them to `[B, V, T]` allows direct comparison with `[B, T]` targets.


In [6]:
class_second_logits = vocabulary_logits.transpose(1, 2)
higher_rank_loss = torch.nn.functional.cross_entropy(
    class_second_logits, target_token_ids
)
higher_rank_position_losses = torch.nn.functional.cross_entropy(
    class_second_logits,
    target_token_ids,
    reduction="none",
)

print("class-last GPT logits:", vocabulary_logits.shape)
print("class-second logits:", class_second_logits.shape)
print("higher-rank targets:", target_token_ids.shape)
print("higher-rank position losses:", higher_rank_position_losses.shape)
print("flattened and higher-rank means match:", end=" ")
print(torch.allclose(mean_loss, higher_rank_loss))
print("position losses match:", end=" ")
print(torch.allclose(loss_per_position, higher_rank_position_losses))

class-last GPT logits: torch.Size([2, 4, 10])
class-second logits: torch.Size([2, 10, 4])
higher-rank targets: torch.Size([2, 4])
higher-rank position losses: torch.Size([2, 4])
flattened and higher-rank means match: True
position losses match: True


Both forms compute the same loss.

This course uses flattening because `[B × T, V]` makes the classification-example interpretation especially clear.


## Interpret the loss scale

If all `V` logits are equal, softmax assigns probability `1 / V` to every class.

The cross-entropy for every target is then `-log(1 / V) = log(V)`.

This uniform-prediction loss is a useful reference value, not a universal performance threshold.


In [7]:
uniform_logits = torch.zeros_like(vocabulary_logits)
uniform_loss = torch.nn.functional.cross_entropy(
    uniform_logits.reshape(-1, vocabulary_size),
    flattened_targets,
)
expected_uniform_loss = torch.log(torch.tensor(float(vocabulary_size), device=device))

print("uniform cross-entropy:", uniform_loss.item())
print("log(vocabulary size):", expected_uniform_loss.item())
print("uniform result matches log(V):", end=" ")
print(torch.allclose(uniform_loss, expected_uniform_loss))

uniform cross-entropy: 2.3025853633880615
log(vocabulary size): 2.3025851249694824
uniform result matches log(V): True


A model that assigns more probability to the correct targets can reduce loss below this reference.

A confidently wrong model can produce loss far above it.


## Correct confidence and wrong confidence

Cross-entropy rewards high target-class logits and penalizes high logits for incorrect classes.

The following controlled tensors differ only in which class receives a large score.


In [8]:
confident_correct_logits = torch.zeros_like(flattened_logits)
confident_wrong_logits = torch.zeros_like(flattened_logits)
row_indexes = torch.arange(flattened_targets.numel(), device=device)
wrong_class_ids = (flattened_targets + 1) % vocabulary_size
confident_correct_logits[row_indexes, flattened_targets] = 5.0
confident_wrong_logits[row_indexes, wrong_class_ids] = 5.0
correct_confidence_loss = torch.nn.functional.cross_entropy(
    confident_correct_logits, flattened_targets
)
wrong_confidence_loss = torch.nn.functional.cross_entropy(
    confident_wrong_logits, flattened_targets
)

print("uniform loss:", uniform_loss.item())
print("confident-correct loss:", correct_confidence_loss.item())
print("confident-wrong loss:", wrong_confidence_loss.item())

uniform loss: 2.3025853633880615
confident-correct loss: 0.058873943984508514
confident-wrong loss: 5.058874130249023


The high correct-class scores sharply reduce loss, while equally large wrong-class scores increase it.


## Pass raw logits, not probabilities

`torch.nn.functional.cross_entropy` expects raw logits.

Applying softmax first changes the function being computed because cross-entropy will treat those probabilities as if they were logits and apply log-softmax again.

The built-in logits path is also more numerically stable than manually combining softmax and logarithms.


In [9]:
flattened_probabilities = torch.softmax(flattened_logits, dim=-1)
incorrect_probability_input_loss = torch.nn.functional.cross_entropy(
    flattened_probabilities, flattened_targets
)

print("raw-logit loss:", mean_loss.item())
print("incorrect probability-input loss:", end=" ")
print(incorrect_probability_input_loss.item())
print("values differ:", end=" ")
print(not torch.allclose(mean_loss, incorrect_probability_input_loss))

raw-logit loss: 3.159121513366699
incorrect probability-input loss: 2.3264362812042236
values differ: True


The different value confirms that pre-applying softmax is not a harmless formatting step.


## Build a reusable loss helper

A helper can enforce the rank, shape, dtype, and target-range contracts before computing loss.

The helper keeps vocabulary as the final class dimension and combines all preceding example dimensions.


In [10]:
def compute_gpt_cross_entropy_loss(
    vocabulary_logits: torch.Tensor,
    target_token_ids: torch.Tensor,
) -> torch.Tensor:
    if vocabulary_logits.ndim != 3:
        raise ValueError("logits must have shape [batch, context, vocabulary].")
    if target_token_ids.ndim != 2:
        raise ValueError("targets must have shape [batch, context].")

    batch_size, context_length, vocabulary_size = vocabulary_logits.shape
    if target_token_ids.shape != (batch_size, context_length):
        raise ValueError("targets must match the logits batch and context.")
    if target_token_ids.dtype not in (torch.int32, torch.int64):
        raise TypeError("targets must contain integer class indexes.")
    if target_token_ids.numel() == 0:
        raise ValueError("targets cannot be empty.")
    if target_token_ids.min().item() < 0:
        raise ValueError("target IDs cannot be negative.")
    if target_token_ids.max().item() >= vocabulary_size:
        raise ValueError("a target ID is outside the vocabulary.")

    flattened_logits = vocabulary_logits.reshape(-1, vocabulary_size)
    flattened_targets = target_token_ids.reshape(-1)
    return torch.nn.functional.cross_entropy(flattened_logits, flattened_targets)


helper_loss = compute_gpt_cross_entropy_loss(vocabulary_logits, target_token_ids)

print("helper loss:", helper_loss.item())
print("helper matches direct loss:", torch.allclose(helper_loss, mean_loss))

helper loss: 3.159121513366699
helper matches direct loss: True


The validation errors would identify malformed inputs before PyTorch reports a less specific shape or class-index failure.


## Diagnose shape mistakes without triggering errors

A safe diagnostic checks each contract before calling cross-entropy.

Flattening all three logit dimensions destroys the per-example class vectors, while leaving targets unflattened produces mismatched example shapes.


In [11]:
all_scores_in_one_vector = vocabulary_logits.reshape(-1)
copy_of_unflattened_targets = target_token_ids.clone()
invalid_float_targets = target_token_ids.to(torch.float32)
out_of_range_targets = target_token_ids.clone()
out_of_range_targets[0, 0] = vocabulary_size

print("correct flattened logits shape:", flattened_logits.shape)
print("all scores in one vector:", all_scores_in_one_vector.shape)
print("correct flattened targets shape:", flattened_targets.shape)
print("unflattened targets shape:", copy_of_unflattened_targets.shape)
print("correct target dtype:", target_token_ids.dtype)
print("invalid target dtype:", invalid_float_targets.dtype)
print("largest valid target ID:", vocabulary_size - 1)
print("invalid target ID:", out_of_range_targets[0, 0].item())

correct flattened logits shape: torch.Size([8, 10])
all scores in one vector: torch.Size([80])
correct flattened targets shape: torch.Size([8])
unflattened targets shape: torch.Size([2, 4])
correct target dtype: torch.int64
invalid target dtype: torch.float32
largest valid target ID: 9
invalid target ID: 10


The correct targets are integer class indexes in the inclusive range `0` through `V - 1`.


## Connect to `TinyGPT`

Inside the Chapter 68 model, the same loss calculation is:

```python
vocabulary_logits, _ = model(input_token_ids)

loss = compute_gpt_cross_entropy_loss(
    vocabulary_logits,
    target_token_ids,
)
```

The architecture supplies `[B, T, V]` logits, and the Chapter 69 batch builder supplies aligned `[B, T]` targets.

The next training loop will backpropagate this scalar mean loss.


## Common mistakes

- Do not flatten vocabulary into the examples dimension.
- Do not flatten logits without flattening targets consistently.
- Do not pass `[B, T, V]` directly when vocabulary is in the last dimension.
- Do not apply softmax before `cross_entropy`.
- Do not use floating-point target IDs.
- Do not use target IDs outside `0` through `V - 1`.
- Do not compute only final-position loss unless that is a deliberate objective.
- Do not assume `view` will work after every transpose or non-contiguous slice.


## Takeaways

GPT logits have shape `[B, T, V]`, and shifted targets have shape `[B, T]`.

The clearest course pattern combines batch and context into one examples dimension.

```python
flattened_logits = vocabulary_logits.reshape(-1, vocabulary_size)
flattened_targets = target_token_ids.reshape(-1)
loss = torch.nn.functional.cross_entropy(
    flattened_logits,
    flattened_targets,
)
```

Each position's loss is the negative log-probability assigned to its target token.

The default scalar loss is the mean over all `B × T` positions.

> Keep vocabulary as the class dimension, and pass raw logits rather than probabilities.


## What comes next

The next chapter will place this scalar loss inside a complete optimization loop with gradient clearing, backpropagation, and parameter updates.

That loop will reveal whether repeated updates reduce next-token loss on training and validation batches.
